# Milestone 3 — Baseline evaluation (Gemma 4 E4B)

1. **Runtime → Change runtime type → T4 GPU**
2. **Run all cells in order** (top to bottom)

Outputs: `predictions_baseline.jsonl` and `results/baseline.json`

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft trl scikit-learn huggingface_hub

In [ ]:
# Clone repo (skip if already cloned)
import os
if not os.path.exists('/content/clinical-code-extractor'):
    !git clone https://github.com/namitrathod/clinical-code-extractor.git
%cd /content/clinical-code-extractor

In [ ]:
import sys, json, time
from pathlib import Path

ROOT = Path('/content/clinical-code-extractor')
sys.path.insert(0, str(ROOT))

from src.validate import load_whitelists

GOLD = ROOT / 'data/test.jsonl'
OUT = ROOT / 'predictions_baseline.jsonl'
MODEL_ID = 'google/gemma-4-E4B-it'

print('ROOT:', ROOT)
print('Test notes:', sum(1 for _ in open(GOLD)))

In [ ]:
# Accept Gemma license on huggingface.co first, then paste your HF token
from huggingface_hub import login
login()

# Inference is run via scripts/run_baseline.py in the next cell (loads the model on GPU).

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

icd_w, cpt_w = load_whitelists(ROOT / 'codes')
rows = load_jsonl(GOLD)
print('Notes:', len(rows))

In [ ]:
!python scripts/run_baseline.py --gold data/test.jsonl --output predictions_baseline.jsonl --max-new-tokens 256

In [ ]:
!python scripts/eval.py --predictions predictions_baseline.jsonl --gold data/test.jsonl --output results/baseline.json

In [ ]:
from google.colab import files
files.download('predictions_baseline.jsonl')
files.download('results/baseline.json')